# Model comparison

Compares the geoai models and simple CVA to the torchgeo models. 
The models were already tested in change_detection_mini.ipynb. Here they are run on the test area and model performance is evaluated systematically

In [1]:
# Setup

from pathlib import Path

PROJECT_DIR = Path.cwd()
PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

STUDY_AREA = "Nepal, Syabru Besi"
TARGET_CRS = "EPSG:32645"  # UTM zone 45N

print(f"Project directory: {PROJECT_DIR}")
print(f"Study area: {STUDY_AREA}")
print(f"Target CRS: {TARGET_CRS}")

Project directory: c:\Users\herol\OneDrive - Universität Zürich UZH\sds320\project
Study area: Nepal, Syabru Besi
Target CRS: EPSG:32645


In [ ]:
import geoai

from geoai.change_detection import (
    ChangeStarDetection,
    changestar_detect,
    list_changestar_models,
)

models = list_changestar_models()
for short_name, full_name in models.items():
    print(f"{short_name:30s} -> {full_name}")

In [2]:
#create RGB images from Sentinel-2 bands for geoai model input
from pathlib import Path
import rasterio

base = PROCESSED_DATA_DIR / "s2_sub_test"
folders = ["t1", "t2"]
bands = ["B04.tif", "B03.tif", "B02.tif"]  # R, G, B

for folder_name in folders:
    folder = base / folder_name
    out_path = folder / f"RGB_{folder_name}.tif"

    with rasterio.open(folder / bands[0]) as ref:
        profile = ref.profile.copy()
    profile.update(count=3)

    with rasterio.open(out_path, "w", **profile) as dst:
        for i, name in enumerate(bands, start=1):
            with rasterio.open(folder / name) as src:
                dst.write(src.read(1), i)
                dst.set_band_description(i, Path(name).stem)

    print(f"Saved {out_path}")

path_before = PROCESSED_DATA_DIR / "s2_sub_test" / "t1" / "RGB_t1.tif"
path_after = PROCESSED_DATA_DIR / "s2_sub_test" / "t2" / "RGB_t2.tif"

Saved c:\Users\herol\OneDrive - Universität Zürich UZH\sds320\project\data\processed\s2_sub_test\t1\RGB_t1.tif
Saved c:\Users\herol\OneDrive - Universität Zürich UZH\sds320\project\data\processed\s2_sub_test\t2\RGB_t2.tif


## ChangeStar s9

In [ ]:
MODEL_RESULTS_DIR = RESULTS_DIR / "model_comparison" / "s9"
MODEL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

detector_s9 = ChangeStarDetection(model_name="s9_s9c1_vitb")
result_s9 = detector_s9.predict(    
    path_before,
    path_after,
    output_change=str(MODEL_RESULTS_DIR / "change_map_s9.tif"),
    output_t1_semantic=str(MODEL_RESULTS_DIR / "t1_semantic_s9.tif"),
    output_t2_semantic=str(MODEL_RESULTS_DIR / "t2_semantic_s9.tif"),
    output_vector=str(MODEL_RESULTS_DIR / "changes_s9.gpkg"),
    tile_size=256, #fix tile size to avoid issues due to non-square input tiff files
    overlap=64
)

In [ ]:
fig = detector_s9.visualize(
    path_before,
    path_after,
    result=result_s9,
    figsize=(25, 10),
    title1="Sentinel-2 Before",
    title2="Sentinel-2 After",
)
plt.show()

In [ ]:
# Evaluate the model's performance using confusion matrix, precision, and recall
from pathlib import Path
import numpy as np
import rasterio
from sklearn.metrics import confusion_matrix, precision_score, recall_score

gt_path = PROCESSED_DATA_DIR / "s2_sub_test" / "mask" / "scene001.tif"
pred_path = MODEL_RESULTS_DIR / "change_map_s9.tif"

with rasterio.open(gt_path) as src:
    gt = src.read(1)
    gt_nodata = src.nodata

with rasterio.open(pred_path) as src:
    pred = src.read(1)
    pred_nodata = src.nodata

assert gt.shape == pred.shape, f"Shape mismatch: {gt.shape} vs {pred.shape}"

# Ignore nodata pixels in either mask
valid = np.ones(gt.shape, dtype=bool)
if gt_nodata is not None:
    valid &= gt != gt_nodata
if pred_nodata is not None:
    valid &= pred != pred_nodata

y_true = (gt[valid] > 0).astype(np.uint8)
y_pred = (pred[valid] > 0).astype(np.uint8)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)

print("Confusion matrix (rows = truth, columns = prediction):")
print(cm)
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
